[Sutton & Barto RL Book]: http://incompleteideas.net/book/RLbook2020.pdf


# Policy Gradient Methods

In policy gradient (PG) methods, the policy does not consult
the action values in its decision of action. Here, a parametrized policy is
learned which selects the action without consulting the estimated action value.
The value function may still be learned, with the aim of learning the policy
(parameters), but it is not specifically consulted in order to take the
action.

## Notations
The following notations will be used [Sutton & Barto RL Book]

$$
\begin{align*}
\mathbf{\theta} \in \mathbb{R} ^{d^{'}} &\quad \text{policy parameter vector} \\[0.5em]
\pi(a | s, \mathbf{\theta}) = \text{Pr}\{A_t = a | S_t, \mathbf{\theta}_t = \mathbf{\theta} \} &\quad \text{action selection probability at time $t$ given state $s$ and parameter $\mathbf{\theta}$} \\[0.5em]
\mathbf{w} \in \mathbb{R}^{d} &\quad \text{value function ($\hat{v}(\cdot, \mathbf{w})$ or $\hat{q}(\cdot, \cdot, \mathbf{w})$) weight vector (or parameters), if the method uses it} \\[0.5em]
J(\mathbf{\theta}) \in \mathbb{R} &\quad \text{scalar performance measure w.r.t the policy parameters}
\end{align*}
$$

## Policy Gradient Methods Overview
In PG methods, learning the policy parameter is based on the  gradient of the
scalar performance measure $J(\mathbf{\theta})$, wherein these methods aim to
_maximize_ its value, via gradient _ascent_ of $J$:

$$
\mathbf{\theta}_{t+1} = \mathbf{\theta}_{t} + \alpha \widehat{\nabla_{\mathbf{\theta}_t} J(\mathbf{\theta}_t)}
$$

where
$\widehat{\nabla_{\mathbf{\theta}_t} J(\mathbf{\theta}_t)} \in \mathbb{R}^{d ^{'}}$
is a stochastic _estimate_ whose _expectation_ approximates the _gradient_
of the performance measure $J$ with respect to the policy parameters $\mathbf{\theta}$.

All PG methods follow this general schema - independent of whether they learn
a state/action value function or not. Methods which do learn value functions
are usually called _actor-critic_, where actor refers to the policy and the
critic the state or (most often) action value function.

## Policy Approximation
In PG methods, the policy can be parametrized in any way, as long as for
$\pi(a | s, \mathbf{\theta})$ there is a gradient wrt its parameters; i.e.
as long as $\nabla_{\mathbf{\theta}}\pi(a | s, \mathbf{\theta})$ exists and is
finite for all $s \in \mathcal{S}$ and $a \in \mathcal{A}$. Typically
the policy never becomes deterministic - $\pi(\cdot) \in \{0, 1\}$,
in order to ensure exploration.

### Discrete Action Space
If the action space is discrete and not too large, one way to parametrize
the policy is via numerical preferences $h(s, a, \mathbf{\theta}) \in \mathbb{R}$.

Actions with the highest preference - in each state - are given higher
probabilities of being selected (remember, we typically do not use
deterministic policies). One way to achieve this is via the softmax function:

$$
\pi(a' | s, \mathbf{\theta} ) = \frac{e^{h(s, a', \mathbf{\theta})}}{\sum_{a \in \mathcal{A}}e^{h(s, a, \mathbf{\theta})}}
$$

This is called _soft-max in action preferences_. Action preferences (function) can
be parameterized in an arbitrary way; for example ANNs or linear features
$h(s, a, \mathbf{\theta}) = \mathbf{\theta}^{T} \mathbf{x}(s, a)$.

###### Approaching determinism
One advantage of parametrizing policies in soft-max in action preferences is that
the policy can approach deterministic policy, whereas with $\varepsilon$-greedy
there's always a chance of selecting a random action with $\varepsilon$
probability.

###### Couldn't we select an action according to the softmax on the action values ?
This approach wouldn't allow the policy to approach determinisim. Action-value
estimates would converge to their corresponding true values, which would
differ by a finite amount, translating to specific probabilities other
than 0 and 1. Action preferences are driven to produce the optimal stochastic
policy (via performance measure maximization). If the optimal policy is
deterministic, then the preferences of the optimal actions will be driven
infinitely higher than all suboptimal actions, as allowed by the parameters
(the probability distribution becomes "peaky" over the course of optimization).

###### Arbitrary probabilities
Another advantage of parametrizing policies is that it allows
the selection of actions with arbitrary probabilities. In certain problems the
best policy is a "soft" policy. For example, in a card game, you may want the
ability to bluff, due to the partial information. Action-value methods have
no natural way of finding stochastic optimal policies, whereas policy
approximating methods can.

###### Flexibility in complexity
One more advantage is advantage that policy parameterization may have
over action-value parameterization is that the policy may be a simpler
function to approximate. Problems vary in the complexity of their policies
and action-value functions. For some, the action-value function is simpler
and thus easier to approximate. For others, the policy is simpler.
In the latter case a policy-based method will typically learn faster and
yield a superior asymptotic policy.

###### Leveraging prior knowledge
The choice of policy parameterization is sometimes a good way
of injecting prior knowledge about the desired form of the policy into
the reinforcement learning system. This is often the most important reason
for using a policy-based learning method. For example, one can leverage
"inductive bias" from the problem at hand in formulating their policy.


## Policy Gradient Theorem
With continuous policy parameterization the action probabilities change
smoothly as a function of the learned parameter. The continuity of the policy dependence on the parameters that enables
policy-gradient methods to approximate gradient ascent. For the episodic
task, the performance measure to be optimized (maximized) is defined
as the value of the start state of the episode:

$$
J(\mathbf{\theta}) \overset \cdot{=} v_{\pi_{\mathbf{\theta}}}(s_0)
$$

where $\pi_{\mathbf{\theta}}$ is the true value function of $\pi_{\mathbf{\theta}}$
parametrized by $\mathbf{\theta}$. In this discussion $\gamma = 1$, i.e.
the undiscounted episode. With function approximation it may seem challenging to change the policy parameter
in a way that ensures improvement:
* Performance depends on both action selection and distribution of states where those actions are taken.
* Action selection depend on the parameters of the policy.
* The effect of the policy on the state distribution is a function of the environment - think the underlying MDP - and it is (typically) unknown.

_How can we estimate the performance gradient with respect to the policy
parameter when the gradient depends on the unknown effect of policy changes
on the state distribution?_

Policy gradient theorem, provides an analytic expression for the gradient of
performance with respect to the policy parameter - and it does not involve
the derivative of the state distribution.

But before we focus on the theorem, we should explain and define its building
blocks (from previous chapters in the book and otherwise).


We have the following identities:
$$
\begin{align*}
q_{\pi}(s, a) &= \sum_{s', r} p(s', r | s, a)(r + v(s')) \\[0.5em]
v_{\pi} &= \sum_a \pi(a | s) q_{\pi}(s, a) \\[0.5em]
p(s' | s, a) &= \sum_r p(s', r | s, a) \\[0.5em]
\end{align*}
$$

Also, the initial state distribution is defined as:
$$
h(s) = \text{Pr{$S_0 = s$}}
$$

The average number of time *steps* spent in state $s$, in a single episode,
is denoted as $\eta(s)$. it includes both if the episode starts in
$s$, and the transitions that are made into $s$ from a preceding
state $\bar{s}$.

$$
\eta(s) = \text{(1-step)} h(s) + \gamma \sum_{\bar{s}} \eta(\bar{s}) \sum_a p(s| \bar{s}, a) \pi(a | \bar{s}) =  h(s) + \gamma \sum_{\bar{s}} \eta(\bar{s})p(s | \bar{s})
$$

_Quick aside_: The discount term $\gamma$ can be thought of as the probability
that the next step is __not__ terminal. Consequentially, $(1 - \gamma)$ is the
probability that the next step is terminal. So in the definition above,
$\gamma$ down–weights the “future‐step visits” term as if the episode might
end with probability $1 - \gamma$

The on-policy distribution ($\mu$) - known as the stationary distribution for
the continuing case - defined as:

$$
\mu(s) = \frac{\eta(s)}{\sum_{s'}\eta(s')}
$$

means the fraction of time spent in each state, normalized.

Moreover, for a discrete Markov chain, as referenced [here](https://en.wikipedia.org/wiki/Discrete-time_Markov_chain)
in the $n$-step transition section, the probability of going from state i
to state j in n time steps is:

$$
p_{ij}^{(n)} = \text{Pr}\{ X_{n} = j | X_0 = i \}
$$

which in the book is defined as the probability of going from $i$ to $j$, in
$n$ steps in the underlying MDP under policy $\pi$, and is defined as:

$$
\text{Pr}\{ i \rightarrow j, n, \pi \} = \text{Pr}_{\pi}\{X_{n} = j | X_{0} = i \}
$$

$n$-step distribution satisfy the Chapman-Kolmogorov equation:

$$
p^{(n)}_{ij} = \sum_r p_{ir}^{(k)}p_{rj}^{n-k} = \sum_{r^{(1)}, r^{(2)}, ... r^{(n-1)}} p_{ir^{(1)}}^{(1)}p_{r^{(1)}r^{(2)}}^{(1)} ... p_{r^{(n-2)}r^{(n-1)}}^{(1)}*p_{r^{(n-1)}j}^{(1)}
$$

Moreover, using an indicator function $I_{k}^{x} = \mathbf{1}\{S_k = x\}$
we can also define the expected count as:

$$
\mathbb{E}[\sum_{k=0} ^ {\infty}I_{k}^{x}] = \sum_{k=0} ^ {\infty} \mathbb{E}[I_{k}^{x}] = \sum_{k=0} ^ {\infty} \text{Pr}\{S_k = x \}
$$

So the expected state visitation count $\eta(s)$ above can also be expressed as:

$$
\begin{align*}
\eta(s) &= \sum_{k=0}^{\infty}\text{Pr}\{s' \rightarrow s, k, \pi \} \\[0.5em]
&= \sum_{k}p_{s's}^{(k)} \\
&= \mathbb{E}[\text{count} \{t: S_t = s \}]
\end{align*}
$$

###### Gradient of the value function

So now we derive the gradient of the value function
($\nabla \overset \cdot{=} \nabla_{\mathbf{\theta}}$ below):

$$
\begin{align*}
\nabla v_{\pi}(s) &= \nabla \Bigl(\sum_a \pi(a|s) q_{\pi}(s, a) \Bigr) \\[0.5em]
&=\sum_a \bigl(\nabla \pi(a | s) q_{\pi}(s, a) + \pi(a | s)\nabla q_{\pi}(s, a)\bigr) \\[0.5em]
&= \sum_a \Bigl(\nabla \pi(a | s) q_{\pi}(s, a) + \pi(a | s)\nabla \bigl(\sum_{s', r} p(s', r | s, a)(r + v(s')) \bigr)\Bigr) \\[0.5em]
&= \sum_a \Bigl(\nabla \pi(a | s) q_{\pi}(s, a) + \pi(a|s) \bigl(\sum_{s'} p(s' | s, a) \nabla v(s') \bigr) \Bigr) \\[0.5em]
&= \sum_a \nabla \pi(a | s) q_{\pi}(s, a) + \sum_a \sum_{s'}\pi(a | s)p(s'|s, a) \nabla v(s') \\[0.5em]
&= \sum_a \nabla \pi(a | s) q_{\pi}(s, a) + \sum_{s'} p(s' | s) \nabla v(s') \\[0.5em]
&= \sum_a \nabla \pi(a | s) q_{\pi}(s, a) + \sum_{s'} p(s' | s) \Bigl[\sum_{a'} \nabla \pi(a' | s') q(s', a') + \sum_{s''} p(s'' | s') \bigl[\sum_{a''}\nabla \pi(a'' | s'') q(s'', a'') + \sum_{s'''} p(s''' | s'')[ ...] \bigr] \Bigr] \\[0.5em]
&= \sum_a \nabla \pi(a | s) q_{\pi}(s, a) + \sum_{s'} p(s' | s)\sum_{a'} \nabla \pi(a' | s') q(s', a') +  \sum_{s'} p(s' | s) \sum_{s''}p(s'' | s') \sum_{a''} \nabla \pi(a'' | s'')q(s'', a'') + \sum_{s'} p(s' | s) \sum_{s''}p(s'' | s') \sum_{s'''}p(s''' | s'')\sum_{a'''}\nabla \pi(a''' | s''')q(s''', a''') + ... \\[0.5em]
&= \sum_a \nabla \pi(a | s) q_{\pi}(s, a) + \sum_{a'}\sum_{s'}p(s' | s) \nabla \pi(a' | s')q(s', a') + \sum_{a''} \sum_{s', s''}p(s'|s)p(s''| s') \nabla \pi(a'' | s'') q(s'', a'') + \sum_{a'''} \sum_{s', s'', s'''}p(s'|s)p(s'' | s)p(s''' | s'') \nabla \pi(a''' | s''')q(s''', a''') + ... \\[0.5em]
&= \sum_a \sum_{s}p(s|s) \nabla \pi(a | s) q_{\pi}(s, a) + \sum_{a'}\sum_{s'} p(s'|s) \nabla \pi(a' | s')q(s', a') + \sum_{a''} \sum_{s''}p(s'' | s)\nabla \pi(a'' | s'')q(s'', a'') + \sum_{a'''}\sum_{s'''}p(s''' | s) \nabla \pi(a''' | s''')q(s''', a''') + ... \quad \text{by Chapman-Kolmogorov equation}  \\[0.5em]
&= \sum_{a \in \mathcal{A}} \sum_{k = 0}^{\infty} \sum_{s^{(k)} \in \mathcal{S}} p(s^{(k)} | s) \nabla\pi(a | s^{(k)})q_{\pi}(s^{(k)}, a)  \quad \text{since $\sum_{a^{(k)}}\pi(a^{(k)} | s^{(k)})$} = \sum_a \pi(a | s^{(k)})\\[0.5em]
&= \sum_{a \in \mathcal{A}, s^{*} \in \mathcal{S}} \sum_{k = 0}^{\infty} p_{ss^{*}} ^ {(k)}\nabla\pi(a | s^{*})q_{\pi}(s^{*}, a) \quad \text{using the standard n-step notation}\\[0.5em]
&= \sum_{a \in \mathcal{A}, s^{*} \in \mathcal{S}} \eta(s^{*})\nabla\pi(a | s^{*})q_{\pi}(s^{*}, a) \\[0.5em]
&= \sum_{s^{*} \in \mathcal{S}} \eta(s^{*})\sum_{a \in \mathcal{A}} \nabla\pi(a | s^{*})q_{\pi}(s^{*}, a)
\end{align*}
$$

where $s^{(k)}$ is the state at $k$-th steps from $s$

###### The theorem, finally
$$
\begin{align*}
\nabla J(\mathbf{\theta}) &= \nabla v_{\pi}(s_0) \\[0.5em]
&= \sum_{s \in \mathcal{S}} \eta(s)\sum_{a \in \mathcal{A}} \nabla\pi(a | s)q_{\pi}(s, a)\\[0.5em]
&= \sum_{s'}\eta(s')\sum_{s \in \mathcal{S}} \frac{\eta(s)}{\eta(s')} \sum_{a \in \mathcal{A}} \nabla\pi(a | s)q_{\pi}(s, a)\\[0.5em]
&= \sum_{s'}\eta(s')\sum_{s \in \mathcal{S}} \mu(s) \sum_{a \in \mathcal{A}} \nabla\pi(a | s)q_{\pi}(s, a)\\[0.5em]
&\propto \sum_{s \in \mathcal{S}} \mu(s) \sum_{a \in \mathcal{A}} \nabla\pi(a | s)q_{\pi}(s, a)
\end{align*}
$$


So, in conclusion, the PG theorem states that the gradient of the
objective/performance function is proportional to the expectation wrt the
state distribution (note that $\pi(\cdot | \cdot) = \pi(\cdot | \cdot, \mathbf{\theta})$:

$$
\fbox{$J(\mathbf{\theta}) \propto \sum_{s \in \mathcal{S}} \mu(s) \sum_{a \in \mathcal{A}} \nabla\pi(a | s, \mathbf{\theta})q_{\pi}(s, a)$}
$$

which allows us to use the samples induced from the policy to obtain
the excpectation of the sample gradient, proportional to the actual gradient.
In the episodic case, the constant of proportionality is the
average length of an episode, and in the continuing case it is 1, so that the relationship is
actually an equality.


## REINFORCE: The Monte Carlo Policy Gradient
The policy gradient theorem gives an exact expression proportional to
the gradient; all that is needed is some way of sampling
whose expectation equals or approximates this expression.
The right-hand side of the policy gradient theorem is a sum over states
weighted by how often the states occur under the target policy $\pi$. If
$\pi$ is followed, then states will be encountered in these proportions.

$$
\begin{align*}
J(\mathbf{\theta}) &\propto \sum_{s \in \mathcal{S}} \mu(s) \sum_{a \in \mathcal{A}} \nabla\pi(a | s)q_{\pi}(s, a) \\[0.5em]
&= \mathbb{E}_{S_t \sim \mu, \; \pi}\Bigl[\sum_{a \in \mathcal{A}} \nabla\pi(a|S_t) q_{\pi}(S_t, a) \Bigr]
\end{align*}
$$

###### All-Actions algorithm

Using this result, we can perform _stochastic_ gradient ascent on the objective function
$J$ using the following update:

$$
\mathbf{\theta}_{t+1} = \mathbf{\theta}_t + \alpha \sum_a \nabla \pi(S_t, a, \mathbf{\theta}) \hat{q}(S_t, a, \mathbf{w})
$$

###### REINFORCE
For reinforce we want to introduce the sampled action $A_t$, same as we did
above for $S_t$, to derive the expectation under $\pi$.

$$
\begin{align*}
\nabla J(\mathbf{\theta}) & \propto \mathbb{E}_{S_t \sim \mu, \; \pi}\Bigl[\sum_{a \in \mathcal{A}} \nabla\pi(a|S_t) q_{\pi}(S_t, a) \Bigr] \\[0.5em]
& = \mathbb{E}_{S_t \sim \mu, \; \pi}\Bigl[\sum_{a \in \mathcal{A}} \pi(a, S_t) \frac{\nabla\pi(a, S_t)}{\pi(a, S_t)}  q_{\pi}(S_t, a) \Bigr] \\[0.5em]
&= \mathbb{E}_{S_t \sim \mu, \; \pi} \Bigl[\sum_{a \in \mathcal{A}} \pi(a | S_t) \nabla \log \pi(a | S_t) q_{\pi}(S_t, a) \Bigr] \\[0.5em]
&= \mathbb{E}_{S_t \sim \mu, \; A_t \sim \pi} \Bigl[\nabla \log \pi(A_t | S_t) q_{\pi}(S_t, A_t) \Bigr] \quad \text{using the sampled actions} \\[0.5em]
&= \mathbb{E}_{S_t \sim \mu, \; A_t \sim \pi} \Bigl[\nabla \log \pi(A_t | S_t) \mathbb{E}_{\pi}[G_t | S_t, A_t] \Bigr] \\[0.5em]
&= \mathbb{E}_{S_t \sim \mu, \; A_t \sim \pi} \Bigl[G_t \nabla \log \pi(A_t | S_t) \Bigr]
\end{align*}
$$

where $G_t$ is the usual trace sum of (discounted if $0 < \gamma < 1$) returns
from the trace. This is what makes REINFORCE an MC method - we are not using
bootstrapping to generate the target return.
All the components of the final form can be fully generated by
samples. As it is shown, the result is proportional to the gradient of the
objective function $J$. The stochastic gradient ascent is thus defined as follows:

$$
\begin{align*}
\mathbf{\theta}_{t+1} &= \mathbf{\theta}_t + \alpha G_t \nabla \log \pi(A_t, S_t) \\[0.5em]
&= \mathbf{\theta}_t + \alpha G_t \frac{\nabla \pi(A_t, S_t)}{\pi(A_t, S_t)}
\end{align*}
$$

The gradient (vector) of the probability of taking the actually taken action
is divided by the probability of taking that action. The vector is the
direction in parameter space that most increases the probability of
repeating the action $A_t$ on future visits to state $S_t$. The update
increases the parameter vector in this direction:
* proportional to the return, favoring highest returns - if positive increase $J$, otherwise decrease.
* inversely proportional to the action probability, favor exploration - prefer less frequent actions

Note that REINFORCE uses the complete return from time $t$, which includes all
future rewards up until the end of the episode. In this sense REINFORCE is a Monte
Carlo algorithm and is well defined only for the episodic case with all updates made in
retrospect after the episode is completed.

<img src="images/reinforce.png" alt="Grid" width="450"/>


###### REINFORCE with Baseline

The PG theorem can be generalized to include a baseline $b(s)$:

$$
\nabla J(\mathbf{\theta}) \propto \sum_s \mu(s) \sum_a \Bigl(q_{\pi}(s, a) - b(s) \Bigr) \nabla \pi(a | s, \mathbf{\theta})
$$

The subtraction of the baseline does not change the gradient expectation because:

$$
b(s) \sum_a \nabla \pi(a | s, \mathbf{\theta}) = bs(s) \nabla \sum_a \pi(a | s, \mathbf{\theta}) = b(s) \nabla 1 = 0
$$

but it's used to reduce the gradient variance, thus speeding up learning. The
baseline can be any function that does not vary with the action.

The new update rule for REINFORCE with baseline is:

$$
\mathbf{\theta}_{t+1} = \mathbf{\theta}_t + \alpha \Bigl(G_t - b(S_t) \Bigr) \nabla \log \pi(A_t | S_t, \mathbf{\theta})
$$

A common choice for the baseline is the state value function
$\hat{v}(S_t, \mathbf{w}_t)$ where the parameter $\mathbf{w} \in \mathbb{R}^d$
iss learned and updated like in the linear approximation methods (refer to
those methods). So in this case we would have 2 sets of parameters
we're learning, $\mathbf{\theta}$ and $\mathbf{w}$.

Because REINFORCE is a Monte-Carlo method, i.e. we use complete
returns generated _at the end of the episode_, to learn the policy parameters
$\mathbf{\theta}$, we can also use the same method for learning the parameters
of the state value function $\mathbf{w}$.

<img src="images/reinforce_with_baseline.png" alt="Grid" width="450"/>


## Actor - Critic Methods
In REINFORCE with baseline, we use a state value function (estimate) of the
state _before_ the action is taken, i.e. $S_t$. This estimate sets a baseline
for the ensuing return $R_{t+1}$, but it cannot be used to evaluate
the action $A_t$. In Actor-Critic (AC) methods we evaluate the following
state ($S_{t+1}$) as well. The estimated value of the second state, when
discounted and added to the reward, constitutes the one-step return $G_{t:t+1}$
which is a useful estimate of the actual return and thus is a way of
assessing the action. With this formulation, we can modulate the bias furthermore
by using $n$-step returns and eligibility traces. When the
state-value function is used to assess actions in this way it is called a
_critic_, and the overall policy-gradient method is termed an _actor–critic_
method.

###### One-Step Actor–Critic Methods
They are the analog of the TD methods, such as TD(0), Sarsa(0), and Q-learning.
They are fully online and incremental, yet avoid the complexities of
eligibility traces.

One-step actor–critic methods replace the full return of REINFORCE with the
one-step estimate of the return, and use a learned state value function as
the baseline.

$$
\begin{align*}
\mathbf{\theta}_{t+1} &= \mathbf{\theta}_{t} + \alpha \Bigl(G_t - \hat{v}(S_t, \mathbf{w}) \Bigr) \nabla_{\mathbf{\theta_t}} \log \pi(A_t | S_t, \mathbf{\theta}_t) \\[0.5em]
&= \mathbf{\theta}_{t} + \alpha \Bigl(R_{t + 1} + \gamma \hat{v}(S_{t+1}, \mathbf{w}) -  \hat{v}(S_{t}, \mathbf{w}) \Bigr) \nabla_{\mathbf{\theta_t}} \log \pi(A_t | S_t, \mathbf{\theta}_t) \\[0.5em]
&= \mathbf{\theta}_{t} + \alpha \delta_t \nabla_{\mathbf{\theta}_t} \log \pi(A_t| S_t, \mathbf{\theta}_t) \\[0.5em]
&= \mathbf{\theta}_{t} + \alpha \delta_t \frac{\nabla_{\mathbf{\theta}_t} \pi(A_t | S_t, \mathbf{\theta}_t)}{\pi(A_t | S_t, \mathbf{\theta}_t)}\\[0.5em]
\end{align*}
$$


The pseudo-code for the episododict algorithm is shown below.
It is fully online, incremental algorithm, with states,
actions, and rewards processed as they occur and then never
revisited.

<img src="images/OneStep_AC.png" alt="Grid" width="450"/>

The implementation is in `agents/OneStepAC`


###### n-Step with Eligibility Traces Actor–Critic
We can replace the one-step target $G_t$ with $G_{t:t+n}$ or the lambda return
$G_t^{\lambda}$. Then using the eligibility traces we can incorportate these
methods into the algorithm below.

<img src="images/AC_with_eligibility_traces.png" alt="Grid" width="450"/>



## Policy Gradient for Continuing Problems
For the continuing setting we need to redefine the objective function (
performance function) in terms of the average rate of reward per step:

$$
\begin{align*}
J(\mathbf{\theta}) \overset \cdot{=} r(\pi) &\overset \cdot{=} \lim_{h \rightarrow \infty} \frac{1}{h} \sum_{t=1}^{h} \mathbb{E}[R_t | S_0, A_{0:t-1} \sim \pi] \\[0.5em]
&= \lim_{t \rightarrow \infty} \mathbb{E}[R_t | S_0, A_{0:t-1} \sim \pi] \\[0.5em]
&= \sum_s \mu(s) \sum_a \pi(a | s) \sum_{s', r} p(s', r | s, a) r
\end{align*}
$$

where $\mu(s)$ is the stationary / steady-state distribution under $\pi$:

$$
\begin{align*}
\mu(s') &\overset \cdot{=} \lim_{t \rightarrow \infty}[S_t = s' | A_{0:t} \sim \pi] \quad \text{independent of $S_0$, ergodic assumption} \\[0.5em]
&= \sum_{s \in \mathcal{S}} \mu(s)\sum_a \pi(a | s)p(s' | s, a) \quad \text{if you follow $\pi$ you remain in the same distribution}
\end{align*}
$$

the pseudo-code of which is shown below:

<img src="images/AC_with_eligibility_traces_continuing.png" alt="Grid" width="450"/>


## Policy Parameterization for _Continuous Actions_

For continuous actions (i.e. infinite actions) we learn the statistics of the
distribution of said action. For example, the action set might be the real numbers, with actions chosen
from a normal (Gaussian) distribution. To produce a policy parameterization,
the policy can be defined as the normal probability density over a
real-valued scalar action, with mean and standard deviation given by
parametric function approximators that depend on the state.

$$
\pi(a | s, \mathbf{\theta}) \overset \cdot{=} \frac{1}{\sigma(s, \mathbf{\theta})} \exp \Bigl(- \frac{\bigl(a - \mu(s, \mathbf{\theta}) \bigr)^2}{2\sigma(s, \mathbf{\theta})^2} \Bigr)
$$


where: $\pi: \mathcal{S} \times\mathbb{R} ^{d'} \rightarrow \mathbb{R}$
and $\sigma: \mathcal{S} \times\mathbb{R} ^{d'} \rightarrow \mathbb{R}^{+}$
are two _parametrized function approximators_. We divide the policy parameters
into the part to be used for the approximation of mean and the other for that
of the standard deviation,
$\mathbf{\theta} \overset \cdot{=} [\mathbf{\theta}_{\pi}, \mathbf{\theta}_{\sigma}]$.

Note that the standard deviation must be positive, and it's best approximated
as the exponential of the function approximator
$\sigma(s, \mathbf{\theta}) = \exp \Bigl( f(\mathbf{x}(s), \mathbf{\theta}_{\sigma}) \Bigr)$,
where $f$ is your logit-function approximator of choice, and $\mathbf{x}$ is the
feature representation of state $s$.

With these definitions, all the algorithms in this section can be used to
learn continuous action selection.
